# AIC26 BTC Keyframe Deduplication — Kaggle Runner

This notebook clones the codebase from GitHub and uses `phash_dedup` (dHash) and `quality_scores` from `pipelines.preprocessing` to remove near-duplicate keyframes from the official BTC keyframe dataset on Kaggle.

**Input:** BTC Keyframes Dataset (`/kaggle/input/...`)
**Output:** `/kaggle/working/deduped_keyframes`

In [ ]:
# --- 1. CLONE CODE FROM GITHUB ---
import os, sys, shutil

GIT_REPO_URL = "https://github.com/Hoaiduc195/aic2026.git"
GIT_BRANCH = "main"
REPO_DIR = "/kaggle/working/aic2026"

shutil.rmtree(REPO_DIR, ignore_errors=True)  # Always fetch fresh code
print(f"Cloning fresh code from GitHub: {GIT_REPO_URL} (branch: {GIT_BRANCH})")
!git clone --depth 1 -b {GIT_BRANCH} {GIT_REPO_URL} {REPO_DIR}

# Add repository root to sys.path so we can import pipelines.preprocessing
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Code successfully cloned and sys.path updated!")

In [ ]:
# --- 2. CONFIGURATION & PATH DISCOVERY ---
import glob

# Try common Kaggle input dataset candidate paths or search recursively for keyframes
CANDIDATE_PATHS = [
    "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/map-keyframes",
    "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/features/map-keyframes",
    "/kaggle/input/btc-keyframes",
    "/kaggle/input/keyframes"
]

INPUT_KEYFRAMES_DIR = None
for cand in CANDIDATE_PATHS:
    if os.path.exists(cand) and glob.glob(f"{cand}/*"):
        INPUT_KEYFRAMES_DIR = cand
        break

if not INPUT_KEYFRAMES_DIR:
    hits = glob.glob("/kaggle/input/**/keyframes", recursive=True)
    if hits:
        INPUT_KEYFRAMES_DIR = hits[0]
    else:
        INPUT_KEYFRAMES_DIR = "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/map-keyframes"

OUTPUT_KEYFRAMES_DIR = "/kaggle/working/deduped_keyframes"
MAX_HAMMING = 5  # dHash Hamming distance threshold (< 5 = duplicate)

print(f"INPUT_KEYFRAMES_DIR  : {INPUT_KEYFRAMES_DIR}")
print(f"OUTPUT_KEYFRAMES_DIR : {OUTPUT_KEYFRAMES_DIR}")

In [ ]:
# --- 3. DEDUPLICATION SCRIPT ---
import cv2
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# Import pipeline functions from cloned repo
from pipelines.preprocessing.keyframes.dedup import phash_dedup
from pipelines.preprocessing.keyframes.quality import quality_scores

NUM_WORKERS = os.cpu_count() or 4

def process_single_video(video_dir_path):
    video_id = os.path.basename(video_dir_path)
    output_video_dir = os.path.join(OUTPUT_KEYFRAMES_DIR, video_id)
    
    image_paths = sorted(
        glob.glob(os.path.join(video_dir_path, "*.jpg")) +
        glob.glob(os.path.join(video_dir_path, "*.png")) +
        glob.glob(os.path.join(video_dir_path, "*.webp"))
    )
    
    if not image_paths:
        return video_id, 0, 0
    
    items = []
    for img_path in image_paths:
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        scores = quality_scores(img_rgb)
        items.append({
            "path": img_path,
            "frame": img_rgb,
            "blur_score": scores["blur_score"]
        })
    
    kept_items = phash_dedup(items, max_hamming=MAX_HAMMING)
    
    os.makedirs(output_video_dir, exist_ok=True)
    for item in kept_items:
        src_path = item["path"]
        dst_path = os.path.join(output_video_dir, os.path.basename(src_path))
        shutil.copy2(src_path, dst_path)
        
    return video_id, len(image_paths), len(kept_items)

video_dirs = [p for p in glob.glob(os.path.join(INPUT_KEYFRAMES_DIR, "*")) if os.path.isdir(p)]
print(f"Found {len(video_dirs)} video directories to process with {NUM_WORKERS} CPU workers.")

os.makedirs(OUTPUT_KEYFRAMES_DIR, exist_ok=True)
total_raw = 0
total_kept = 0

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_single_video, v_dir): v_dir for v_dir in video_dirs}
    for future in tqdm(as_completed(futures), total=len(video_dirs), desc="Deduplicating Keyframes"):
        try:
            vid, raw_cnt, kept_cnt = future.result()
            total_raw += raw_cnt
            total_kept += kept_cnt
        except Exception as e:
            print(f"Error processing {futures[future]}: {e}")

reduction = 100.0 * (1.0 - total_kept / total_raw) if total_raw else 0
print("\n" + "="*50)
print(f"DEDUPLICATION COMPLETED!")
print(f"Raw BTC keyframes  : {total_raw:,}")
print(f"Kept keyframes     : {total_kept:,}")
print(f"Reduction          : -{reduction:.2f}% ({total_raw - total_kept:,} frames removed)")
print(f"Output directory   : {OUTPUT_KEYFRAMES_DIR}")
print("="*50)

In [ ]:
# --- 4. SUMMARY & NEXT STEPS ---
print("Dataset ready at /kaggle/working/deduped_keyframes")
print("1. Click 'Save Version' -> 'Save & Run All (Commit)'.")
print("2. Go to the Output tab of the committed version.")
print("3. Click 'Create Dataset' on deduped_keyframes to publish the clean keyframe dataset!")